# Lab 10 Challenge: worked solution

The challenge has two tasks: ask Puja Srivastava's question on the graph built by hand in Part D, and reject a refund
request with a reason. This notebook runs on its own. It repeats the config cell, the model, the three lookups and
their tools from the lab, then Part D's graph for task 1 and Part F's refund table, write tool and approval
middleware for task 2.

## Setup: the pieces from the lab

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The model, the read-only connection to the shop's database, the three lookups and their tools, as in Parts A and B.

In [2]:
import json
import sqlite3
from langchain_openai import ChatOpenAI
from langchain.tools import tool

llm = ChatOpenAI(model=MODEL, reasoning_effort=EFFORT)
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list:
    """Find a shop customer by full name. Returns a list of matches with their customer_id."""
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0]}


TOOLS = [tool(find_customer), tool(get_invoices), tool(get_support_rep)]
SYSTEM = ("You help the support reps of a music shop. Look up every fact with the tools; never answer from memory. "
          "Answer in one or two sentences.")
print(find_customer("Puja Srivastava"), get_invoices(59), get_support_rep(59))

[{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}] {'customer_id': 59, 'invoice_count': 6, 'total': 36.64} {'customer_id': 59, 'support_rep': 'Jane Peacock'}


The right answers for Puja, from the lookups: customer 59, 6 invoices, 36.64 US dollars, Jane Peacock.

## Task 1: Puja's question on the graph built by hand

Part D's graph, unchanged: the state with the `add_messages` reducer, the model node with the tools bound, the
`route` function, `ToolNode` and the back edge.

In [3]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


model_with_tools = llm.bind_tools(TOOLS)


def model_node(state):
    reply = model_with_tools.invoke([SystemMessage(SYSTEM)] + state["messages"])
    return {"messages": [reply]}


def route(state):
    if state["messages"][-1].tool_calls:
        return "tools"
    return END


builder = StateGraph(AgentState)
builder.add_node("model", model_node)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", route, ["tools", END])
builder.add_edge("tools", "model")
hand_graph = builder.compile()
print(list(hand_graph.get_graph().nodes))

['__start__', 'model', 'tools', '__end__']


In [4]:
PUJA_QUESTION = "How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?"
puja_result = hand_graph.invoke({"messages": [{"role": "user", "content": PUJA_QUESTION}]})
for message in puja_result["messages"]:
    if message.type == "ai" and message.tool_calls:
        print("ai     asks for", [(call["name"], call["args"]) for call in message.tool_calls])
answer = puja_result["messages"][-1].text
print("answer:", answer)
for fact in ("6", "36.64", "Jane Peacock"):
    print("✅" if fact in answer else "❌", fact)

ai     asks for [('find_customer', {'name': 'Puja Srivastava'})]
ai     asks for [('get_invoices', {'customer_id': 59}), ('get_support_rep', {'customer_id': 59})]
answer: Puja Srivastava has 6 invoices totaling $36.64. Her support rep is Jane Peacock.
✅ 6
✅ 36.64
✅ Jane Peacock


The graph built by hand ran the same three steps for Puja as for Helena: `find_customer`, then `get_invoices` and `get_support_rep` together with customer 59, then the answer, *Puja Srivastava has 6 invoices totaling $36.64. Her support rep is Jane Peacock.* All three facts are in it. The answer is read with `.text`: the model node's replies are lists of content blocks, as in Part C. The
check `"6" in answer` is loose (a 6 anywhere would pass), so read the sentence too; here the 6 is the invoice count.

## Task 2: Reject a refund with a reason

Part F's refund table, write tool and approval middleware. This notebook keeps its table in a file of its own,
`outputs/challenge_refund_requests.db` in the lab's folder (gitignored), and empties it first.

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

OUTPUTS = ROOT / "10_LangGraph" / "outputs"
OUTPUTS.mkdir(exist_ok=True)
refunds = sqlite3.connect(OUTPUTS / "challenge_refund_requests.db", check_same_thread=False)
refunds.execute("CREATE TABLE IF NOT EXISTS refund_requests "
                "(request_id INTEGER PRIMARY KEY, invoice_id INTEGER, amount REAL, reason TEXT)")
refunds.execute("DELETE FROM refund_requests")
refunds.commit()


def file_refund_request(invoice_id: int, amount: float, reason: str) -> str:
    """File a refund request for one invoice, in US dollars. A person reviews it before any money moves."""
    cursor = refunds.execute("INSERT INTO refund_requests (invoice_id, amount, reason) VALUES (?, ?, ?)",
                             (invoice_id, amount, reason))
    refunds.commit()
    return f"refund request {cursor.lastrowid} filed: invoice {invoice_id}, {amount} US dollars"


def show_requests():
    rows = refunds.execute("SELECT * FROM refund_requests").fetchall()
    print("refund_requests:", rows or "empty")


approval = HumanInTheLoopMiddleware(interrupt_on={
    "file_refund_request": {"allowed_decisions": ["approve", "edit", "reject"]}})
desk = create_agent(llm, TOOLS + [tool(file_refund_request)], system_prompt=SYSTEM,
                    middleware=[approval], checkpointer=InMemorySaver())
show_requests()

refund_requests: empty


The request, on its own thread. The run should stop before the write.

In [6]:
REQUEST = ("Puja Srivastava was charged twice for invoice 229 (13.86 US dollars). "
           "Please file a refund request for the duplicate charge.")
config = {"configurable": {"thread_id": "challenge-reject"}}
paused = desk.invoke({"messages": [{"role": "user", "content": REQUEST}]}, config, version="v2")
print("waiting on:", [stop.value["action_requests"][0] for stop in paused.interrupts])
show_requests()

waiting on: [{'name': 'file_refund_request', 'args': {'invoice_id': 229, 'amount': 13.86, 'reason': 'Duplicate charge; customer reports being charged twice for invoice 229.'}, 'description': "Tool execution requires approval\n\nTool: file_refund_request\nArgs: {'invoice_id': 229, 'amount': 13.86, 'reason': 'Duplicate charge; customer reports being charged twice for invoice 229.'}"}]
refund_requests: empty


The run stopped before the write, waiting on `file_refund_request` with invoice 229 and 13.86 US dollars, the numbers in the request, and the table is empty. Now the reviewer's decision: `reject`, with a `message` that tells the model what happened and what to
do next. The model sees only what the middleware passes on, so the message is the only way it learns why.

In [7]:
decision = {"type": "reject",
            "message": "Rejected: billing shows one charge for invoice 229, not two. Do not file it again; tell the rep."}
done = desk.invoke(Command(resume={"decisions": [decision]}), config, version="v2")
tool_message = [m for m in done.value["messages"] if m.type == "tool"][-1]
print("tool status:", tool_message.status)
print("tool text:  ", tool_message.text)
print("reply:      ", done.value["messages"][-1].text)
print("interrupts: ", done.interrupts)
show_requests()

tool status: error
tool text:   User rejected the tool call for `file_refund_request` with reason: Rejected: billing shows one charge for invoice 229, not two. Do not file it again; tell the rep.
reply:       I couldn’t file the refund request: billing shows only one charge for invoice 229, not two.
interrupts:  ()
refund_requests: empty


The tool message has `status` `error` and reads *User rejected the tool call for `file_refund_request` with reason:* followed by our message. The model's reply: *I couldn’t file the refund request: billing shows only one charge for invoice 229, not two.* No interrupt is left, and the table is still empty: no row for invoice 229.

What each part shows:

- The **tool message** is how the rejection reaches the model: the middleware wrote it in place of the tool's result,
  with `status` `error`, so the model knows the call did not run and why.
- The **reply** passes the reason on to the rep. Its wording comes from the model; the facts in it come from our
  message.
- The **table** is the system of record: no row for invoice 229, because `file_refund_request` never ran.